In [1]:
#Install all dependency packages for the course
#Remember to execute this before running any of the exercises
!pip install tenacity==9.0.0
!pip install langchain==0.3.12
!pip install langchain-openai==0.2.12
!pip install langchain_community==0.3.12
!pip install langgraph==0.2.59
!pip install pysqlite3-binary==0.5.4
!pip install langchain_chroma==0.1.4
!pip install pandas==2.2.3
!pip install pypdf==5.1.0
!pip install nbformat==5.10.4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 37.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 751.2/751.2 kB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 51.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 599.5/599.5 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 644.4/644.4 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 781.7/781.7 kB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 10.8 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 13.4 MB/s eta 0:00:00a 0:00:01
  Installing build dependencies .

## 2.3. Setup Function Tools for ReAct Agent

In [5]:
from langchain_core.tools import tool

#Tool annotation identifies a function as a tool automatically
@tool
def find_sum(x:int, y:int) -> int :
    #The docstring comment describes the capabilities of the function
    #It is used by the agent to discover the function's inputs, outputs and capabilities
    """
    This function is used to add two numbers and return their sum.
    It takes two integers as inputs and returns an integer as output.
    """
    return x + y

@tool
def find_product(x:int, y:int) -> int :
    """
    This function is used to multiply two numbers and return their product.
    It takes two integers as inputs and returns an integer as ouput.
    """
    return x * y

@tool
def find_difference(x:int, y:int) -> int :
    """
    This function is used to subtract two numbers and retrun the diffrence.
    It takes two integers as inputs and return an integer as output.
    """
    return x - y


In [13]:
!pip install langchain-groq

## 2.4. Create a basic ReAct Agent

In [18]:
#from langchain_openai import AzureChatOpenAI
from langchain_groq import ChatGroq
import os
#Setup the LLM for the agent

#API info. Replace with your own keys and end points
#os.environ["AZURE_OPENAI_API_KEY"]="4e4ab31800a64ae892cbb768fe28c0fc"
#os.environ["AZURE_OPENAI_ENDPOINT"]="https://agentic-ai-course-kumaran.openai.azure.com/"


model = ChatGroq(temperature=0, groq_api_key="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", model_name="meta-llama/llama-4-maverick-17b-128e-instruct")


#Setup the LLM
#model = AzureChatOpenAI(
    #azure_deployment="gpt-4o" ,
    #api_version="2023-03-15-preview",
    #model="gpt-4o"
#)

#Test the model
response = model.invoke("Hello, how are you?")
print(response.content)


I'm just a language model, I don't have feelings or emotions like humans do, but I'm functioning properly and ready to help with any questions or tasks you might have! How can I assist you today?


In [19]:
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import AIMessage,HumanMessage,SystemMessage

#Create list of tools available to the agent
agent_tools=[find_sum, find_product, find_difference]

#System prompt
system_prompt = SystemMessage(
    """You are a Math genius who can solve math problems. Solve the
    problems provided by the user, by using only tools available. 
    Do not solve the problem yourself"""
)

agent_graph=create_react_agent(
    model=model, 
    state_modifier=system_prompt,
    tools=agent_tools)


## 2.5. Execute the ReAct Agent

In [25]:
#Example 1
inputs = {"messages":[("user","what is the diffrence of 2 and 3 ?")]}

result = agent_graph.invoke(inputs)

#Get the final answer
print(f"Agent returned : {result['messages'][-1].content} \n")

print("Step by Step execution : ")
for message in result['messages']:
    print(message.pretty_repr())

Agent returned : The difference between 2 and 3 is -1. 

Step by Step execution : 
================================ Human Message =================================

what is the diffrence of 2 and 3 ?
================================== Ai Message ==================================
Tool Calls:
  find_difference (call_q0fe)
 Call ID: call_q0fe
  Args:
    x: 2
    y: 3
================================= Tool Message =================================
Name: find_difference

-1
================================== Ai Message ==================================

The difference between 2 and 3 is -1.


In [21]:
#Example 2
inputs = {"messages":[("user","What is 3 multipled by 2 and 5 + 1 ?")]}

result = agent_graph.invoke(inputs)

#Get the final answer
print(f"Agent returned : {result['messages'][-1].content} \n")

print("Step by Step execution : ")
for message in result['messages']:
    print(message.pretty_repr())

Agent returned : The final answer is 36. 

Step by Step execution : 
================================ Human Message =================================

What is 3 multipled by 2 and 5 + 1 ?
================================== Ai Message ==================================
Tool Calls:
  find_product (call_vnsq)
 Call ID: call_vnsq
  Args:
    x: 3
    y: 2
  find_sum (call_n063)
 Call ID: call_n063
  Args:
    x: 5
    y: 1
================================= Tool Message =================================
Name: find_product

6
================================= Tool Message =================================
Name: find_sum

6
================================== Ai Message ==================================
Tool Calls:
  find_product (call_gz5q)
 Call ID: call_gz5q
  Args:
    x: 3
    y: 2
  find_sum (call_2zjf)
 Call ID: call_2zjf
  Args:
    x: 5
    y: 1
  find_product (call_qtzj)
 Call ID: call_qtzj
  Args:
    x: 6
    y: 6
================================= Tool Message ====================

## 2.6. Debugging the Agent

In [24]:
agent_graph=create_react_agent(
    model=model, 
    state_modifier=system_prompt,
    tools=agent_tools,
    debug=False)

inputs = {"messages":[("user","what is the difference of  2 and 3 ?")]}

result = agent_graph.invoke(inputs)